# 🚦 Routing Agent

### **Overview**
In many agentic systems, **Routing** is a crucial step– It decides *what* a user wants to do before the system actually does it.

In this notebook, we build a **Semantic Router** for a Doctor's Assistant. This agent takes natural language queries (e.g., *"I'm busy tomorrow"*) and maps them to specific, executable API actions (e.g., `BLOCK_SLOTS`).

### **Notebook Structure**
1.  **Action Definition**: Explicitly defining the "Menu of Services" (Intents) available to the user.
2.  **Schema Enforcement**: Using Pydantic to force the LLM to output a strict classification structure.
3.  **The Router Chain**: Building the LangChain pipeline that processes the user query.
4.  **Evaluations**: Running the router against a "Golden Dataset" of historical queries.
5.  **Performance Analysis**: Using Precision/Recall metrics to identify where the router gets confused.

**Run modes:** the default `replay` evaluates the repository's saved router predictions
without an API key. `live` creates fresh OpenAI router predictions. Run this notebook first,
then [Jev alone](./jev_as_judge.ipynb) and [Jev/LLM comparison](./llm_as_judge.ipynb) to evaluate failure judgments and compare costs.
The judge lab uses its separately human-labeled frozen traces; new router predictions
need new human judge labels before they can enter that agreement benchmark.


## 1. Environment and run mode

Replay uses the saved router trace, matched by test ID and input text. Live mode uses
LangChain and an OpenAI key to run the router again. The intent definitions and schema
below are shared by both modes. Outputs go to `.runs/` so source CSVs remain reproducible.

In [1]:
# Run from this directory, the repository root, or a fresh Colab runtime.
import os, sys, subprocess
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_REF = os.environ.get("ROUTING_EVALS_REF", "codex/jev-routing-judge")
candidates = [Path.cwd(), Path.cwd() / "routing_agent_evals",
              Path.cwd() / "AI-Evals-Principles/routing_agent_evals"]
LAB = next((p for p in candidates if (p / "judge_lab.py").exists()), None)
if LAB is None and IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    LAB = checkout / "AI-Evals-Principles/routing_agent_evals"
if LAB is None or not (LAB / "judge_lab.py").exists():
    raise RuntimeError("Open the notebook from its repository checkout; see README for setup.")
LAB = LAB.resolve()
os.chdir(LAB)
sys.path.insert(0, str(LAB))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "scikit-learn", "matplotlib", "python-dotenv"], check=True)

MODE = os.environ.get("ROUTER_MODE", "replay")  #@param ["replay", "live"]
assert MODE in {"replay", "live"}
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from sklearn.metrics import classification_report
import pandas as pd
from prompts.router_prompts import ROUTER_PROMPT_1, ROUTER_PROMPT_2
if MODE == "live":
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langchain", "langchain-openai"], check=True)
    from langchain_core.prompts import PromptTemplate
    from langchain.chat_models import init_chat_model
print("Router mode:", MODE)


Router mode: replay


In [2]:
load_dotenv(LAB / ".env", override=False)
if MODE == "live" and not os.environ.get("OPENAI_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        except Exception:
            pass
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI key (hidden): ").strip()


![Diagram](./images/01_img.png)

## 2. Defining the Intent Space (The "Menu")
An LLM cannot classify intents if it doesn't know what they are. We define an **Enum** containing every possible action the bot can take.

### 🧠 Technical Concept:
Each Enum member contains a **Description**. This description is crucial—it serves as the "Prompt" that tells the LLM what that intent actually means.
* **Granularity**: Notice how we distinguish between `BLOCK_SLOTS` (Doctor unavailable) and `CANCEL_APPOINTMENTS` (Patient cancellation). Clear definitions reduce routing errors.
* **Fallbacks**: We include `UNSUPPORTED_REQUEST` and `IRRELEVANT_QUERY` to safely handle inputs that the bot cannot process.

![Diagram](./images/02_img.png)

In [3]:
from enum import Enum

class DoctorActionCategories(Enum):
    LIST_APPOINTMENTS = ("LIST_APPOINTMENTS", "Use this to view, check, or list existing appointments. This is a read-only action. This tool is for viewing the calendar, not for historical data analysis like counting past patients. Example: 'What does my schedule look like tomorrow?")
    CANCEL_APPOINTMENTS = ("CANCEL_APPOINTMENTS", "Use this to cancel all appointments within a general date/time range. It does NOT support specific cancellations (e.g., 'cancel Aman's appointment'); those are 'unsupportedRequest'. Examples: 'Cancel my appointments tomorrow afternoon', 'Wipe my schedule for the 25th'.")
    RESCHEDULE_APPOINTMENTS = ("RESCHEDULE_APPOINTMENTS", "Use this to move all appointments from one date/time range to another. It does NOT support rescheduling a single, specific patient's appointment; those are 'unsupportedRequest'. Examples: 'Move my 24th's appointments to the 25th', 'Shift tomorrow's schedule to next week'.")
    BLOCK_SLOTS = ("BLOCK_SLOTS", "Use this to block time slots or make the doctor unavailable Examples: 'Block my calendar tomorrow', 'I won't be available from 2-4pm', 'Mark me unavailable next week'.")
    UNBLOCK_SLOTS = ("UNBLOCK_SLOTS", "Use this to unblock time slots or make the doctor available again. Examples: 'Unblock my calendar for tomorrow', 'I'm back from my break', 'I am free now'.")
    UPDATE_SPECIAL_SLOTS = ("UPDATE_SPECIAL_SLOTS", "Use this to make changes to the doctor's standard availability. If the doctor wants to make some changes in his availability such as if he wants to add some hours or wants to reduce the hours of his schedule, use this intent. Examples: 'Change my weekend availability', 'Extend my hours this Friday'.")
    CONVERSATIONAL_GREETING = ("CONVERSATIONAL_GREETING", "Use this for simple, conversation-starting social interactions. This category is only for initial greetings like 'hello', 'hey there', or 'how are you'.")
    CLOSING_CONVERSATION = ("CLOSING_CONVERSATION", "Use this for simple, conversation-ending social interactions. This is only for phrases like 'thank you', 'thanks a lot', 'sounds good', or 'goodbye'.")
    UNSUPPORTED_REQUEST = ("UNSUPPORTED_REQUEST", "Use this for requests related to the schedule that cannot be handled by other tools. Crucially, this includes any request to cancel, move, or reschedule a *specific* appointment identified by patient name or relative position (e.g., 'my next appointment'). Also use for vague requests where the goal isn't clear (e.g., 'lighten my schedule') or for creating new appointments.")
    IRRELEVANT_QUERY = ("IRRELEVANT_QUERY", "This is a final fallback category. Use this ONLY if the user's request is completely unrelated to the doctor, their schedule, patients, or the hospital. Use for off-topic queries like asking for jokes, the weather, or ordering food.")

## 3. Flattening the Intent Definitions
To pass these definitions into the LLM's prompt context, we need to convert our Enum into a list of strings (or tuples) that the model can read.

### 🧠 Code Context
* `action_list`: This extracts the raw values (Name + Description) from the Enum so we can inject them dynamically into the system prompt later.

In [4]:
action_list = [action.value for action in DoctorActionCategories]
action_list

[('LIST_APPOINTMENTS',
  "Use this to view, check, or list existing appointments. This is a read-only action. This tool is for viewing the calendar, not for historical data analysis like counting past patients. Example: 'What does my schedule look like tomorrow?"),
 ('CANCEL_APPOINTMENTS',
  "Use this to cancel all appointments within a general date/time range. It does NOT support specific cancellations (e.g., 'cancel Aman's appointment'); those are 'unsupportedRequest'. Examples: 'Cancel my appointments tomorrow afternoon', 'Wipe my schedule for the 25th'."),
 ('RESCHEDULE_APPOINTMENTS',
  "Use this to move all appointments from one date/time range to another. It does NOT support rescheduling a single, specific patient's appointment; those are 'unsupportedRequest'. Examples: 'Move my 24th's appointments to the 25th', 'Shift tomorrow's schedule to next week'."),
 ('BLOCK_SLOTS',
  "Use this to block time slots or make the doctor unavailable Examples: 'Block my calendar tomorrow', 'I wo

## 4. Model Initialization
We initialize the Chat Model that will power our router.

### 🧠 Technical Concept: Latency vs. Intelligence
For routing tasks, we often use smaller, faster models (like `gpt-4o-mini`) because routing needs to happen quickly (low latency) before the main work begins.
* `temperature=0`: This reduces sampling variation; it does not guarantee identical or correct classifications across runs.

In [5]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0) if MODE == "live" else None

## 5. The Router Agent (Schema Enforcement)
This is the core logic. We define a **Pydantic Schema** that forces the LLM to return a structured object instead of raw text.

### 🧠 Code Context
* `DoctorActionSchema`: The blueprint for the output.
    * `intent`: Restricted to our specific Enum list (Literal).
    * `confidence`: The model self-reports how sure it is (`high`, `medium`, `low`). This allows us to build "Human-in-the-Loop" logic later (e.g., if confidence is low, ask the user to clarify).
    * `reasoning`: A brief explanation for inspecting the prediction. It is not proof that the decision is correct.

![Diagram](./images/03_img.png)

In [6]:
def router_agent(user_query):
    if MODE != "live":
        raise RuntimeError("Use live mode for a fresh query; replay uses saved dataset traces.")
    # Define output strcuture
    class DoctorActionSchema(BaseModel):
        """Structured output for Doctor intent classification"""

        intent: Literal["LIST_APPOINTMENTS", "CANCEL_APPOINTMENTS", "RESCHEDULE_APPOINTMENTS", "BLOCK_SLOTS", "UNBLOCK_SLOTS", "UPDATE_SPECIAL_SLOTS", "CONVERSATIONAL_GREETING", "CLOSING_CONVERSATION", "UNSUPPORTED_REQUEST", "IRRELEVANT_QUERY"] = Field(
            description="The classified Doctor Action Intent"
        )

        confidence: Literal["high", "medium", "low"] = Field(
            description="Confidence level of the classification"
        )

        reasoning: str = Field(
            description="Brief explanation of the classification"
        )

    llm_with_structured_output = llm.with_structured_output(DoctorActionSchema)
    template = PromptTemplate.from_template(ROUTER_PROMPT_1)

    chain = template | llm_with_structured_output

    action_list = [action.value for action in DoctorActionCategories]

    result = chain.invoke({'user_query': user_query, 'action_list': action_list})

    return result



## 6. Unit Testing (Sanity Check)
Before processing the full dataset, we run a single test case to verify the router works.

### 🧠 Code Context
* **Input**: *"Can you cancel all my appointments for today?"*
* **Expected Output**: `CANCEL_APPOINTMENTS` with `high` confidence.

In [7]:
if MODE == "live":
    display(router_agent('Can you cancel all my appointments for today?'))
else:
    display(pd.read_csv(LAB / "data/golden_dataset_with_predictions.csv").head(1))
    print("Saved trace shown; no fresh inference in replay mode.")

,TEST_ID,USER_INPUT,TARGET_INTENT,final_intent,final_confidence,final_reasoning,intent_match
0,TEST_001,List my schedule for this afternoon.,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,The user is asking to view their schedule for ...,pass


Saved trace shown; no fresh inference in replay mode.


## 7. Batch Evals (Golden Dataset)
To trust this router in production, we need to test it against a **Golden Dataset** a collection of user queries with **Ground Truth** labels (the correct answers).

### 🧠 Code Context
* `pd.read_csv`: Loads the test data.
* `df.apply(...)`: Runs the `router_agent` function on every query in the dataset.
* **intent_match**: We create a new column to instantly see if the Router's prediction (`final_intent`) matches the Human Ground Truth (`TARGET_INTENT`).

![Diagram](./images/04_img.png)

In [8]:
dataset_path = LAB / "data/golden_dataset.csv"
output_path = LAB / ".runs/router_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
golden = pd.read_csv(dataset_path)
assert golden.TEST_ID.is_unique
if MODE == "live":
    df = golden.copy()
    results = df["USER_INPUT"].apply(router_agent)
    df["final_intent"] = results.apply(lambda result: result.intent)
    df["final_confidence"] = results.apply(lambda result: result.confidence)
    df["final_reasoning"] = results.apply(lambda result: result.reasoning)
else:
    saved = pd.read_csv(LAB / "data/golden_dataset_with_predictions.csv")
    assert saved.TEST_ID.is_unique and set(saved.TEST_ID) == set(golden.TEST_ID)
    df = golden.merge(saved.drop(columns="TARGET_INTENT"), on=["TEST_ID", "USER_INPUT"],
                      validate="one_to_one")
    assert len(df) == len(golden), "Saved inputs differ; regenerate or reconcile the references."
    target_comparison = golden.merge(saved[["TEST_ID", "TARGET_INTENT"]], on="TEST_ID")
    different = target_comparison.TARGET_INTENT_x != target_comparison.TARGET_INTENT_y
    print(f"{different.sum()} saved target labels differ. Metrics use the canonical golden_dataset.csv.")
df["intent_match"] = df.final_intent.eq(df.TARGET_INTENT).map({True: "pass", False: "fail"})
display(df.head())
print("Rows:", len(df), "| routing accuracy:", df.intent_match.eq("pass").mean())


2 saved target labels differ. Metrics use the canonical golden_dataset.csv.


,TEST_ID,USER_INPUT,TARGET_INTENT,final_intent,final_confidence,final_reasoning,intent_match
0,TEST_001,List my schedule for this afternoon.,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,The user is asking to view their schedule for ...,pass
1,TEST_002,What are my appointments Tuesday morning?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,The user is asking to view their appointments ...,pass
2,TEST_003,What's on my calendar this week?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,The user is asking to view their schedule for ...,pass
3,TEST_004,What is my availability Monday morning?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,"The user is asking about their availability, w...",pass
4,TEST_005,View my schedule New Years Eve,LIST_APPOINTMENTS,LIST_APPOINTMENTS,high,The user is asking to view their schedule for ...,pass


Rows: 301 | routing accuracy: 0.7940199335548173


## 8. Saving Results
We save the predictions to a CSV file. This is useful for offline analysis or for using the **LLM-as-a-Judge** workflow to evaluate the reasoning quality later.

In [9]:
df.to_csv(output_path, index=False)

## 9. Performance Evaluation (Metrics)
Now we analyze the results. We load the processed dataset to calculate how well our router performed.

In [10]:
df = pd.read_csv(output_path)

## 10. Class Distribution Analysis
We analyze the balance of our dataset.

### 🧠 Why this matters?
If 90% of our test data is `LIST_APPOINTMENTS`, a model could simply guess "List Appointments" every time and look accurate, while completely failing at critical tasks like `CANCEL_APPOINTMENTS`. Knowing the distribution helps us interpret the metrics correctly.

In [11]:
#Lets look at class distributions
class_distribution = (
    df["TARGET_INTENT"].value_counts()
    .rename_axis("intent")
    .reset_index(name="count")
    .assign(percentage=lambda x: (x["count"] / x["count"].sum() * 100).round(2))
    .sort_values("percentage", ascending = False)
)
class_distribution

,intent,count,percentage
0,BLOCK_SLOTS,59,19.60
1,RESCHEDULE_APPOINTMENTS,51,16.94
2,LIST_APPOINTMENTS,46,15.28
3,UNBLOCK_SLOTS,40,13.29
4,UPDATE_SPECIAL_SLOTS,34,11.30
5,CANCEL_APPOINTMENTS,32,10.63
6,UNSUPPORTED_REQUEST,19,6.31
7,IRRELEVANT_QUERY,8,2.66
8,CONVERSATIONAL_GREETING,6,1.99
9,CLOSING_CONVERSATION,6,1.99


## 11. Classification Report (Precision & Recall)
This is the "Report Card" for our Router. We calculate key metrics for **each intent**.

### 🧠 Metric Definitions
* **Precision**: *"When the model predicts X, is it actually X?"*
    * *Low Precision* = High False Positives (e.g., routing a greeting to "Cancel Appointment").
* **Recall**: *"Out of all the actual X queries, how many did the model find?"*
    * *Low Recall* = High False Negatives (e.g., missing a "Cancel" request and treating it as "Unsupported").
* **F1-Score**: The harmonic mean of Precision and Recall.

### 🔍 Analysis Tip
Look closely at **confusable intents**. For example, does the model confuse `BLOCK_SLOTS` (Doctor unavailable) with `UNBLOCK_SLOTS` (Doctor available)? Low scores in these rows indicate the Prompt Description needs refinement.

In [12]:
#Precision/Recall per class

class_report = classification_report(
    df["TARGET_INTENT"],
    df["final_intent"],
    output_dict=True,
    zero_division=0,
)

pd.DataFrame(class_report).T

,precision,recall,f1-score,support
BLOCK_SLOTS,0.923077,0.813559,0.864865,59.00000
CANCEL_APPOINTMENTS,0.909091,0.625000,0.740741,32.00000
CLOSING_CONVERSATION,1.000000,1.000000,1.000000,6.00000
CONVERSATIONAL_GREETING,1.000000,1.000000,1.000000,6.00000
IRRELEVANT_QUERY,0.470588,1.000000,0.640000,8.00000
LIST_APPOINTMENTS,0.954545,0.913043,0.933333,46.00000
RESCHEDULE_APPOINTMENTS,1.000000,0.705882,0.827586,51.00000
UNBLOCK_SLOTS,0.800000,0.900000,0.847059,40.00000
UNSUPPORTED_REQUEST,0.326923,0.894737,0.478873,19.00000
UPDATE_SPECIAL_SLOTS,0.952381,0.588235,0.727273,34.00000


### 🚀 Next Steps & Optimization
Now that we have baseline metrics, here are a few things we can consider improve the Router:

1.  **Prompt Engineering**: If specific intents have low scores (e.g., `UNSUPPORTED_REQUEST`), refine their descriptions in the `DoctorActionCategories` Enum.
2.  **Few-Shot Prompting**: Add examples of tricky queries directly into the `ROUTER_PROMPT` to guide the model.
3.  **Confidence Thresholding**: Configure the system to hand off to a human agent if the router's `confidence` score is `low` or `medium`.

### 📚 Resources
* **[LangChain Structured Output](https://python.langchain.com/docs/how_to/structured_output/)**: Guide on forcing JSON schemas from LLMs.
* **[Semantic Routing](https://python.langchain.com/docs/concepts/#routing)**: Conceptual guide on different routing techniques.
* **[Scikit-Learn Classification Report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)**: Documentation for interpreting F1, Precision, and Recall.

## Next: evaluate the evaluator

Start with [Jev alone through TypeSafe](./jev_as_judge.ipynb), which replays the recorded
Jev run without a key. Then continue to [Jev-as-a-Judge and LLM-as-a-Judge](./llm_as_judge.ipynb). The comparison uses the
existing `llm_as_judge_dataset.csv` human labels to compare a categorical Jev judge,
an independent LLM judge, and **Jev first → low confidence → LLM**. A calibration set
selects the confidence threshold; a separate held-out set measures agreement and API cost.
Router intent accuracy and judge agreement are different measurements. The new notebook
explains System 1 thinking, tests whether the LLM fixes uncertain Jev errors, and shows
when escalation costs more or introduces new errors.


In [13]:
assert len(df) == len(golden) == 301
assert df["final_intent"].isin([a.value[0] for a in DoctorActionCategories]).all()
print("Router notebook completed:", MODE)
print("Continue with llm_as_judge.ipynb for the Jev / LLM / cascade comparison.")

Router notebook completed: replay
Continue with llm_as_judge.ipynb for the Jev / LLM / cascade comparison.
